# Imports

In [1]:
import pandas as pd
import sys
from pathlib import Path
import datetime

from dateutil.relativedelta import relativedelta

sys.path.insert(0, str(Path.cwd().parent))

In [2]:
from olympic_data import OlympicData
from filtering import (
    get_medal_winners,
    filter_by_year_range,
    get_first_n_names,
    filter_by_names,
)

from biography_scraper import (
    find_wikidata_entry,
    get_wikidata_claims
)

from processing import convert_dict_to_pd_df, convert_dates

# Functions

In [ ]:
def calculate_age(row, currentdate):
    if pd.notna(row['birthdate']) and pd.notna(row['deathdate']):
        age = relativedelta(row['deathdate'], row['birthdate']).years
    elif pd.notna(row['birthdate']):
        age = relativedelta(currentdate, row['birthdate']).years
    else:
        age = None
    return age

In [ ]:
def find_percentage_correct_names(df):
    missing = (df[['qid']].isnull().sum().sum())
    return (f"{100-(missing/df.shape[0]*100)}%")

# Cleaning dataset

In [3]:
"""Run the Olympic data workflow."""
olympics = OlympicData("../data/athlete_events.csv")
olympics.load_and_prepare()

names = get_first_n_names(
    olympics.data,
    n=2000,
)

In [4]:
name_list = []
for name in names:
    name_list.append(get_wikidata_claims(find_wikidata_entry(name)))

In [5]:
df = convert_dict_to_pd_df(name_list)
df = convert_dates(df)

df_original = filter_by_names(olympics.data, names)

In [6]:
merged_df = df_original.merge(
    df,
    left_on='Name',
    right_on='name',
    how='left'
)
merged_df = merged_df.drop(['name'], axis=1)
merged_df["medals"] = merged_df["Gold"] + merged_df["Silver"] + merged_df["Bronze"]

In [ ]:
currentdate = datetime.date.today()
merged_df["age"] = merged_df.apply(lambda row: calculate_age(row, currentdate), axis=1)
merged_df["age"] = merged_df["age"].astype("Int64")

In [ ]:
# merged_df.to_csv("first2000.csv")

# Exploring

In [72]:
find_percentage_correct_names(df)

'36.4%'

In [74]:
merged_df.head()

,ID,Name,Sex,Age,Height,Weight,Team,NOC,Games,Year,...,Silver,Gold,qid,birthdate,deathdate,status,mannerofdeath,errormessage,medals,age
0,1,A Dijiang,M,24.0,180.0,80.0,China,CHN,1992 Summer,1992,...,0,0,Q600404,1967-08-21,NaT,Alive,None,,0,59
1,2,A Lamusi,M,23.0,170.0,60.0,China,CHN,2012 Summer,2012,...,0,0,Q2107375,1989-06-02,NaT,Alive,None,,0,37
2,3,Gunnar Nielsen Aaby,M,24.0,NaN,NaN,Denmark,DEN,1920 Summer,1920,...,0,0,Q4329216,1895-07-09,1966-08-22,Dead,None,,0,71
3,4,Edgar Lindenau Aabye,M,34.0,NaN,NaN,Denmark/Sweden,DEN,1900 Summer,1900,...,0,1,Q561976,1865-09-14,1941-04-30,Dead,None,,1,75
4,5,Christine Jacoba Aaftink,F,21.0,185.0,82.0,Netherlands,NED,1988 Winter,1988,...,0,0,Q56960,1966-08-25,NaT,Alive,None,,0,60


In [56]:
merged_df.isnull().sum()

ID                  0
Name                0
Sex                 0
Age               190
Height            755
Weight            758
Team                0
NOC                 0
Games               0
Year                0
Season              0
City                0
Sport               0
Event               0
Bronze              0
Silver              0
Gold                0
qid              2265
birthdate        2331
deathdate        3283
status           2432
mannerofdeath    3493
errormessage        0
medals              0
age              2331
dtype: int64

In [57]:
merged_df.shape

(3518, 25)

In [45]:
merged_df[['ID', 'status', 'Year']].groupby(['Year', 'status']).count()

,,ID
Year,status,
1900,Dead,2
1908,Dead,3
1912,Dead,14
1920,Dead,12
1924,Dead,13
1928,Dead,14
1932,Dead,6
1936,Dead,4
1948,Alive,2


In [12]:
get_medal_winners(merged_df)[['Team', 'Gold', 'Silver', 'Bronze']].groupby(['Team']).sum()

,Gold,Silver,Bronze
Team,,,
Argentina,2,1,2
Australia,1,1,4
Austria,0,1,1
Azerbaijan,1,1,1
Belarus,1,0,0
...,...,...,...
United States,23,12,7
United States-1,0,1,0
Uruguay,0,0,1
